# Explore DIGIT’s quantitative results

This walkthrough calls the same functions as the CLI. It consumes precomputed
CSV annotations: no CARLA, GPU, NLP model, or API service is required. Install
`scripts/stats/requirements.txt` plus `nbformat nbclient ipykernel` in this kernel.

The study profile defines the experimental design, analysis settings, and random
seeds. Read `doc/analysis.md` for formulas and assumptions.


In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "scripts/stats/paper.yaml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the DIGIT checkout")
sys.path.insert(0, str(ROOT / "src"))

from stats.config import load_config
from stats.validation import load_results
from stats.calculations import effectiveness
from stats.cli import run_analysis

config = load_config(ROOT / "scripts/stats/paper.yaml")
frame = load_results(config)
frame.groupby(["model", "method"]).size().rename("attempts")


## Inspect effectiveness directly

All attempted tests remain in the denominator. Raw flags are retained, while
refused/errored attempts count as no observed end-to-end success. Scenario means
have equal weight; ± in the publication table is sample SD across repetition
rates, in percentage points.


In [ ]:
effectiveness_tables = effectiveness(frame, config)
effectiveness_tables["table_I"]


In [ ]:
# Baseline’s two rates answer different questions; do not interchange them.
effectiveness_tables["all_model_summary"].loc[
    ["baseline / rule-based"],
    ["mean_adversarial_rate", "mean_repetition_rate", "sd_repetition_rate"]
]


## Generate analysis outputs

The complete workflow writes numerical CSVs, readable tables, figures, and input
provenance. The summary links to those results for you to inspect. Malformed
inputs stop execution, and the source CSVs are never modified.


In [ ]:
OUTPUT = ROOT / "output/walkthrough"
tables, figure_names = run_analysis("reproduce", config, OUTPUT)
print(f"{len(tables)} tables; {len(figure_names)} figures")
print(f"Output summary: {OUTPUT / 'report.md'}")


In [ ]:
tables["table_II"][["schedule", "deadline_hours", "baseline_completed_attempts", "novelty_percent"]]


## Distinguish the evidence

Table II uses each target’s modeled completion deadline. Figure 5 uses one
common deadline. Progression contains observed ordered DIGIT outcomes and exact
conditional baseline expectations; subset ranges are not population CIs. Human
naturalness evidence covers incremental only. Scene mentions are automated
annotations, with flagged subsets rather than extra counts.


In [ ]:
from IPython.display import Image, display
for name in ["figure_4b_failure_scenarios", "figure_6a_sampled_naturalness", "figure_8_scene_relevance"]:
    display(Image(filename=str(OUTPUT / "figures" / f"{name}.png"), width=560))


## Adapt the workflow

Copy the YAML profile and edit inputs, design, and timing assumptions for your
study. Use individual commands for studies without sampled ratings or scene
annotations. Supporting session, repetition-overlap, full-pool, and scheduling
sensitivities are available through `diagnostics`. See the analysis guide for
the full schema and interpretation.
